[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/biology_aging/blob/main/notebooks/build-the-goldilocks-dose.ipynb)

# The Goldilocks Dose: Why "More" Is the Wrong Question 🥣
### A hands-on companion to Chapter 4

Chapter 4 is about the **nutrient-sensing dials** — mTOR, IGF-1, AMPK, sirtuins — that switch your cells between **growing** and **maintaining themselves**. Its biggest surprise is that the interventions which most reliably extend life in animals — **caloric restriction, rapamycin, even exercise** — all work by *mild stress*, and they all follow the same shape: **too little does nothing, too much does harm, and the benefit lives at an intermediate dose.** Biologists call it **hormesis**.

In this notebook you'll:
1. **build** that inverted-U curve from the two opposing forces underneath it,
2. discover why **every intervention has its own sweet spot**, and
3. uncover the deeper reason it all works — the **growth-vs-maintenance trade-off** — and why it means *we may be able to live longer than evolution tuned us to.*

No setup. Runs in Colab. Press ▶ on each cell, top to bottom.

## Step 0 — Tools

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("Ready.")

## Part 1 — Build the curve from two opposing forces

A hormetic intervention does **two things at once**:

- It switches on **protective maintenance** — autophagy, stress defenses, repair — but with *diminishing returns* (the first bit of stress helps a lot; more helps less). We'll model that benefit as a saturating curve: `B(d) = Bmax · d/(Kd + d)`.
- It also does **direct harm** that gets *worse faster* the more you take (toxicity, side effects). We'll model that as: `H(d) = a · d²`.

The **net effect on health** is simply protection minus harm: `B(d) − H(d)`. Let's plot all three and find the best dose.

In [ ]:
dose = np.linspace(0, 6, 1000)
Bmax, Kd, a = 10.0, 1.0, 1.0          # <-- try changing these and re-running

benefit = Bmax * dose/(Kd + dose)     # protective maintenance (saturating)
harm    = a * dose**2                 # direct toxicity (accelerating)
net     = benefit - harm

best = np.argmax(net)
print(f"Best dose: {dose[best]:.2f}   net benefit there: {net[best]:.2f}")
print(f"At zero dose: {net[0]:.2f} (nothing happens).  At a big dose (6): {net[-1]:.2f} (net harm).")

plt.figure(figsize=(7,4.5))
plt.plot(dose, benefit, '--', color="#2a9d8f", label="protection (saturating)")
plt.plot(dose, harm,    '--', color="#e76f51", label="harm (accelerating)")
plt.plot(dose, net,     '-',  color="#264653", lw=2.5, label="NET effect on health")
plt.axhline(0, color="k", lw=0.5)
plt.scatter([dose[best]],[net[best]], color="#264653", zorder=5, s=60)
plt.annotate("the sweet spot", (dose[best], net[best]), textcoords="offset points", xytext=(10,-4))
plt.xlabel("dose  (of CR, rapamycin, exercise, a stressor...)"); plt.ylabel("effect on health")
plt.title("Hormesis: the inverted-U is a tug-of-war"); plt.legend(); plt.tight_layout(); plt.show()

The inverted-U isn't a mysterious law — it **falls out** of two ordinary curves pulling against each other. Where protection's diminishing returns meet harm's accelerating cost, you get a peak. **Zero** does nothing; **too much** tips into net harm; the good stuff lives in between. *That's why the question "should I take more?" is the wrong question — the right one is "what's the dose?"*

## Part 2 — Every intervention has its *own* sweet spot

Now the useful part. How toxic an intervention is (our `a`) completely changes where its sweet spot sits. A **gentle** intervention can be pushed far and still help; a **harsh** one has a narrow, low sweet spot — or none. Watch the optimum move as we crank up the toxicity.

In [ ]:
plt.figure(figsize=(7,4.5))
toxicities = [0.3, 0.7, 1.5, 3.0, 6.0]
opt_dose, opt_benefit = [], []
for a in toxicities:
    net = Bmax*dose/(Kd+dose) - a*dose**2
    j = np.argmax(net)
    opt_dose.append(dose[j]); opt_benefit.append(net[j])
    plt.plot(dose, net, label=f"toxicity a = {a}")
    plt.scatter([dose[j]],[net[j]], zorder=5)
plt.axhline(0, color="k", lw=0.5); plt.ylim(-2, 6)
plt.xlabel("dose"); plt.ylabel("net effect on health")
plt.title("Gentle interventions -> wide, high sweet spot.  Harsh ones -> narrow, low.")
plt.legend(); plt.tight_layout(); plt.show()

fig,ax=plt.subplots(1,2,figsize=(11,3.8))
ax[0].plot(toxicities, opt_dose, 'o-', color="#457b9d"); ax[0].set_xlabel("toxicity a"); ax[0].set_ylabel("best dose"); ax[0].set_title("Harsher -> use less")
ax[1].plot(toxicities, opt_benefit, 'o-', color="#2a9d8f"); ax[1].set_xlabel("toxicity a"); ax[1].set_ylabel("benefit at best dose"); ax[1].set_title("Harsher -> less to gain")
plt.tight_layout(); plt.show()

This is why **exercise and mild caloric restriction** have a wide, forgiving sweet spot, while **megadosing a drug or a supplement backfires** — recall the sharp lesson from Chapter 5, where high-dose antioxidants *erased* the benefits of exercise. Same curve, pushed past its peak. The optimal dose is **specific to the molecule**, which is exactly why "a longevity supplement" with no dose-response data is a red flag.

## Part 3 — The deeper *why*: the growth-vs-maintenance dial

Why does *mild stress* help at all? Chapter 4's answer is the **disposable-soma** idea. An organism has a **limited energy budget**, and it must split it between two jobs:

- **growing and reproducing** now, versus
- **maintaining and repairing** the body for later.

The nutrient-sensing pathways *are* that dial: lots of food and growth signaling (**mTOR high**) → spend on growth; scarcity (**mTOR low**, via CR or rapamycin) → spend on maintenance. Turning the dial toward maintenance buys you a longer life — but the budget is fixed, so you **pay in growth and reproduction.** Let's see the trade-off, and then something surprising.

In [ ]:
f = np.linspace(0, 1, 1000)          # fraction of the budget spent on MAINTENANCE
amp = 3.0                             # how much maintenance can extend life

lifespan      = 1.0*(1 + amp*f)      # more maintenance -> longer life
reproduction  = 1.0*(1 - f)          # ...but less reproduction right now
fitness       = reproduction*lifespan # what evolution 'cares about': lifetime offspring

f_evo  = f[np.argmax(fitness)]       # the dial evolution settles on
f_life = f[np.argmax(lifespan)]      # the dial that would maximize lifespan

fig,ax=plt.subplots(1,2,figsize=(11,4.2))
ax[0].plot(f, lifespan, color="#2a9d8f", lw=2, label="lifespan")
ax[0].plot(f, reproduction, color="#e76f51", lw=2, label="reproduction now")
ax[0].set_xlabel("dial -> maintenance"); ax[0].set_ylabel("relative amount")
ax[0].set_title("No free lunch: buy lifespan, pay in reproduction"); ax[0].legend()

ax[1].plot(f, fitness, color="#264653", lw=2.5)
ax[1].axvline(f_evo,  color="#e76f51", ls="--", label=f"evolution's setting (f={f_evo:.2f})")
ax[1].axvline(f_life, color="#2a9d8f", ls="--", label=f"longest-life setting (f={f_life:.2f})")
ax[1].axvspan(f_evo, f_life, color="#2a9d8f", alpha=0.10)
ax[1].set_xlabel("dial -> maintenance"); ax[1].set_ylabel("evolutionary fitness")
ax[1].set_title("Evolution stops short of the longest life"); ax[1].legend()
plt.tight_layout(); plt.show()

print(f"Evolution sets the dial at f={f_evo:.2f}, but lifespan keeps rising all the way to f={f_life:.2f}.")
print(f"That gap (shaded) is unused longevity -- room a drug like rapamycin could turn the dial into.")

Look at what the model just told you. Evolutionary fitness peaks at an **intermediate** maintenance setting — but **lifespan keeps climbing past it.** Evolution tuned your dial to maximize *reproduction*, not *longevity*, so it stopped short of the longest possible life. Everything to the right of evolution's setting is **unused lifespan.**

That gap is the whole opportunity of this chapter. **Caloric restriction and rapamycin turn the dial rightward** — toward maintenance, past the fitness optimum your genes were set to — buying years by spending a reproductive budget an adult no longer needs. It's the book's *"brake and price"* in one picture: **the maintenance setting really is the longevity setting**, and there's no free lunch. (And the honesty of the chapter still holds — rapamycin is the most reproducible *animal* geroprotector, its side effects *are* the price, and whether the trade pays off in *humans* is not yet settled.)

## Go further (for the reader who finished hungry)

- **Sweep the dial knobs.** Change `amp` in Part 3 (how much maintenance can extend life). When does the gap between evolution's setting and the longest-life setting grow — and when does it vanish?
- **Make harm have a threshold.** Replace `a*d²` with a term that's near-zero until some dose and then shoots up. How does the sweet spot change?
- **Two interventions at once.** Give two hormetic stressors and see whether their sweet spots add up — or whether "stacking" pushes you over the edge.
- **Connect the whole book.** This same inverted-U is the *hormesis* of Chapter 5 (mild mitochondrial stress) and the *"partial"* window of Chapter 7 (reprogramming) — one shape, three chapters, rising stakes. *Biology runs on curves, not lines.*

---
*Companion notebook to* **Chapter 4 — Deregulated Nutrient Sensing**, *from the "AI for Drug Discovery for Kids" aging & longevity volume. A teaching model: the shapes are real, the numbers are illustrative.*